# Data Processing — INX Future Inc. Employee Performance
**IABAC Certified Data Scientist — Project Code: 10281**

This notebook loads the raw INX Future Inc. employee dataset, inspects data quality,
performs cleaning and encoding, and writes a modeling-ready dataset to `data/processed/`.
It also **persists the fitted encoders** (not just their mapping) so that `predict_model.ipynb`
can correctly transform brand-new, raw records — a step the first draft of this project missed.


In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
RAW_PATH = '../../data/raw/INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1_8.xls'
df = pd.read_excel(RAW_PATH)
print('Shape:', df.shape)
df.head()

Shape: (1200, 28)


,EmpNumber,Age,Gender,EducationBackground,MaritalStatus,EmpDepartment,EmpJobRole,BusinessTravelFrequency,DistanceFromHome,EmpEducationLevel,EmpEnvironmentSatisfaction,EmpHourlyRate,EmpJobInvolvement,EmpJobLevel,EmpJobSatisfaction,NumCompaniesWorked,OverTime,EmpLastSalaryHikePercent,EmpRelationshipSatisfaction,TotalWorkExperienceInYears,TrainingTimesLastYear,EmpWorkLifeBalance,ExperienceYearsAtThisCompany,ExperienceYearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition,PerformanceRating
0,E1001000,32,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,10,3,4,55,3,2,4,1,No,12,4,10,2,2,10,7,0,8,No,3
1,E1001006,47,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,14,4,4,42,3,2,1,2,No,12,4,20,2,3,7,7,1,7,No,3
2,E1001007,40,Male,Life Sciences,Married,Sales,Sales Executive,Travel_Frequently,5,4,4,48,2,3,1,5,Yes,21,3,20,2,3,18,13,1,12,No,4
3,E1001009,41,Male,Human Resources,Divorced,Human Resources,Manager,Travel_Rarely,10,4,2,73,2,5,4,3,No,15,2,23,2,2,21,6,12,6,No,3
4,E1001010,60,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,16,4,1,84,3,2,1,8,No,14,4,10,1,3,2,2,2,2,No,3


## 1. Data Quality Check

In [2]:
print('Data types:')
print(df.dtypes)
print()
print('Missing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0] if df.isnull().sum().sum() > 0 else 'No missing values found.')
print()
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate EmpNumber:', df['EmpNumber'].duplicated().sum())

Data types:
EmpNumber                         str
Age                             int64
Gender                            str
EducationBackground               str
MaritalStatus                     str
EmpDepartment                     str
EmpJobRole                        str
BusinessTravelFrequency           str
DistanceFromHome                int64
EmpEducationLevel               int64
EmpEnvironmentSatisfaction      int64
EmpHourlyRate                   int64
EmpJobInvolvement               int64
EmpJobLevel                     int64
EmpJobSatisfaction              int64
NumCompaniesWorked              int64
OverTime                          str
EmpLastSalaryHikePercent        int64
EmpRelationshipSatisfaction     int64
TotalWorkExperienceInYears      int64
TrainingTimesLastYear           int64
EmpWorkLifeBalance              int64
ExperienceYearsAtThisCompany    int64
ExperienceYearsInCurrentRole    int64
YearsSinceLastPromotion         int64
YearsWithCurrManager            int64


In [3]:
# Check cardinality of categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
for c in cat_cols:
    print(f"{c}: {df[c].nunique()} unique -> {df[c].unique()[:8]}")

EmpNumber: 1200 unique -> <StringArray>
['E1001000', 'E1001006', 'E1001007', 'E1001009', 'E1001010', 'E1001011',
 'E1001016', 'E1001019']
Length: 8, dtype: str
Gender: 2 unique -> <StringArray>
['Male', 'Female']
Length: 2, dtype: str
EducationBackground: 6 unique -> <StringArray>
[       'Marketing',    'Life Sciences',  'Human Resources',
          'Medical',            'Other', 'Technical Degree']
Length: 6, dtype: str
MaritalStatus: 3 unique -> <StringArray>
['Single', 'Married', 'Divorced']
Length: 3, dtype: str
EmpDepartment: 6 unique -> <StringArray>
[                 'Sales',        'Human Resources',            'Development',
           'Data Science', 'Research & Development',                'Finance']
Length: 6, dtype: str
EmpJobRole: 19 unique -> <StringArray>
[     'Sales Executive',              'Manager',            'Developer',
 'Sales Representative',      'Human Resources',     'Senior Developer',
       'Data Scientist',   'Senior Manager R&D']
Length: 8, dtype: str


/tmp/ipykernel_88/2098516068.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


**Observations:**
- Dataset has 1200 employee records and 28 columns, with no missing values and no duplicate `EmpNumber` records — no imputation is required.
- `EmpNumber` is a unique identifier and carries no predictive signal, so it is dropped before modeling.
- Categorical columns (Gender, EducationBackground, MaritalStatus, EmpDepartment, EmpJobRole, BusinessTravelFrequency, OverTime, Attrition) need encoding for machine learning models.
- Many "rating-style" columns (EmpEnvironmentSatisfaction, EmpJobInvolvement, EmpJobSatisfaction, EmpWorkLifeBalance, EmpEducationLevel, EmpJobLevel, EmpRelationshipSatisfaction) are already ordinal integers (1–5) and need no further encoding.

## 2. Outlier Inspection (numeric columns)

In [4]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
outlier_summary = {}
for c in numeric_cols:
    q1, q3 = df[c].quantile(0.25), df[c].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = ((df[c] < lower) | (df[c] > upper)).sum()
    outlier_summary[c] = n_out
pd.Series(outlier_summary).sort_values(ascending=False)

PerformanceRating               326
TrainingTimesLastYear           188
YearsSinceLastPromotion          88
ExperienceYearsAtThisCompany     56
TotalWorkExperienceInYears       51
NumCompaniesWorked               39
ExperienceYearsInCurrentRole     16
YearsWithCurrManager             11
EmpEducationLevel                 0
Age                               0
DistanceFromHome                  0
EmpRelationshipSatisfaction       0
EmpLastSalaryHikePercent          0
EmpJobSatisfaction                0
EmpJobLevel                       0
EmpHourlyRate                     0
EmpJobInvolvement                 0
EmpEnvironmentSatisfaction        0
EmpWorkLifeBalance                0
dtype: int64

**Decision:** Columns like `YearsSinceLastPromotion`, `TotalWorkExperienceInYears`, `NumCompaniesWorked` and `TrainingTimesLastYear` show IQR-flagged points, but these are genuine, plausible HR values (e.g. an employee not promoted in 15 years, or an employee with 40 years experience), not data-entry errors. They are retained as-is — tree-based and linear models used later are robust to this kind of natural skew, and removing them would discard real signal about tenure/promotion patterns.

## 3. Encoding Categorical Features

Encoders are **fit once here and persisted** with `joblib`, so every downstream notebook (training, prediction) uses the exact same category → integer mapping. This is what lets `predict_model.ipynb` accept a raw record like `EmpDepartment='Sales'` and transform it correctly, instead of requiring pre-encoded input.

In [5]:
from sklearn.preprocessing import LabelEncoder
import joblib

df_processed = df.drop(columns=['EmpNumber']).copy()

# Binary categorical columns -> explicit maps (saved, not just inline dicts)
binary_maps = {
    'Gender': {'Male': 1, 'Female': 0},
    'OverTime': {'Yes': 1, 'No': 0},
    'Attrition': {'Yes': 1, 'No': 0},
}
for col, mapping in binary_maps.items():
    df_processed[col] = df_processed[col].map(mapping)

# Multi-class nominal columns -> label encode, KEEP the fitted encoder objects
label_encoders = {}
multi_cat_cols = ['EducationBackground', 'MaritalStatus', 'EmpDepartment', 'EmpJobRole', 'BusinessTravelFrequency']
for col in multi_cat_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])
    label_encoders[col] = le  # the fitted object itself, not just its mapping

for col, le in label_encoders.items():
    print(col, '->', dict(zip(le.classes_, le.transform(le.classes_))))

EducationBackground -> {'Human Resources': np.int64(0), 'Life Sciences': np.int64(1), 'Marketing': np.int64(2), 'Medical': np.int64(3), 'Other': np.int64(4), 'Technical Degree': np.int64(5)}
MaritalStatus -> {'Divorced': np.int64(0), 'Married': np.int64(1), 'Single': np.int64(2)}
EmpDepartment -> {'Data Science': np.int64(0), 'Development': np.int64(1), 'Finance': np.int64(2), 'Human Resources': np.int64(3), 'Research & Development': np.int64(4), 'Sales': np.int64(5)}
EmpJobRole -> {'Business Analyst': np.int64(0), 'Data Scientist': np.int64(1), 'Delivery Manager': np.int64(2), 'Developer': np.int64(3), 'Finance Manager': np.int64(4), 'Healthcare Representative': np.int64(5), 'Human Resources': np.int64(6), 'Laboratory Technician': np.int64(7), 'Manager': np.int64(8), 'Manager R&D': np.int64(9), 'Manufacturing Director': np.int64(10), 'Research Director': np.int64(11), 'Research Scientist': np.int64(12), 'Sales Executive': np.int64(13), 'Sales Representative': np.int64(14), 'Senior Dev

In [6]:
df_processed.head()

,Age,Gender,EducationBackground,MaritalStatus,EmpDepartment,EmpJobRole,BusinessTravelFrequency,DistanceFromHome,EmpEducationLevel,EmpEnvironmentSatisfaction,EmpHourlyRate,EmpJobInvolvement,EmpJobLevel,EmpJobSatisfaction,NumCompaniesWorked,OverTime,EmpLastSalaryHikePercent,EmpRelationshipSatisfaction,TotalWorkExperienceInYears,TrainingTimesLastYear,EmpWorkLifeBalance,ExperienceYearsAtThisCompany,ExperienceYearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition,PerformanceRating
0,32,1,2,2,5,13,2,10,3,4,55,3,2,4,1,0,12,4,10,2,2,10,7,0,8,0,3
1,47,1,2,2,5,13,2,14,4,4,42,3,2,1,2,0,12,4,20,2,3,7,7,1,7,0,3
2,40,1,1,1,5,13,1,5,4,4,48,2,3,1,5,1,21,3,20,2,3,18,13,1,12,0,4
3,41,1,0,0,3,8,2,10,4,2,73,2,5,4,3,0,15,2,23,2,2,21,6,12,6,0,3
4,60,1,2,2,5,13,2,16,4,1,84,3,2,1,8,0,14,4,10,1,3,2,2,2,2,0,3


**Why label encoding for these columns:** `EmpDepartment`, `EmpJobRole`, `EducationBackground`, `MaritalStatus` and `BusinessTravelFrequency` are nominal (no inherent order). Label encoding is used here (rather than one-hot) to keep the feature space compact for tree-based models (Random Forest / Gradient Boosting / XGBoost), which handle label-encoded nominal features well by splitting on category groups.

## 4. Which Features Are Actually Knowable About a Hiring Candidate?

The project brief asks for a model that predicts performance **for use in hiring**. A candidate
being screened has no employment history at INX yet, so any feature that only exists *because*
someone already works there cannot be used as model input for that use case — using it would be
a feature-availability error (the model would need information at prediction time that simply
does not exist for a candidate).

Each column is tagged below as `hiring_safe` (knowable about an external candidate/applicant —
demographics, education, prior work history, and the role/department being hired into) or not
(satisfaction/engagement scores, compensation growth, tenure, promotion history, current overtime
status, attrition — all of which only exist for someone already employed).

In [7]:
hiring_safe_features = [
    'Age', 'Gender', 'EducationBackground', 'MaritalStatus',
    'EmpDepartment', 'EmpJobRole', 'EmpJobLevel', 'BusinessTravelFrequency',
    'DistanceFromHome', 'EmpEducationLevel',
    'NumCompaniesWorked', 'TotalWorkExperienceInYears',
]

not_hiring_safe_features = [
    'EmpEnvironmentSatisfaction', 'EmpHourlyRate', 'EmpJobInvolvement', 'EmpJobSatisfaction',
    'EmpRelationshipSatisfaction', 'EmpWorkLifeBalance', 'OverTime', 'EmpLastSalaryHikePercent',
    'TrainingTimesLastYear', 'ExperienceYearsAtThisCompany', 'ExperienceYearsInCurrentRole',
    'YearsSinceLastPromotion', 'YearsWithCurrManager', 'Attrition',
]

all_predictors = [c for c in df_processed.columns if c != 'PerformanceRating']
assert set(hiring_safe_features) | set(not_hiring_safe_features) == set(all_predictors), \
    'Every predictor must be classified as hiring-safe or not.'
assert set(hiring_safe_features) & set(not_hiring_safe_features) == set(), 'No overlap allowed.'

print(f'{len(hiring_safe_features)} hiring-safe features (usable for candidate screening):')
print(hiring_safe_features)
print()
print(f'{len(not_hiring_safe_features)} features only available for existing employees:')
print(not_hiring_safe_features)

12 hiring-safe features (usable for candidate screening):
['Age', 'Gender', 'EducationBackground', 'MaritalStatus', 'EmpDepartment', 'EmpJobRole', 'EmpJobLevel', 'BusinessTravelFrequency', 'DistanceFromHome', 'EmpEducationLevel', 'NumCompaniesWorked', 'TotalWorkExperienceInYears']

14 features only available for existing employees:
['EmpEnvironmentSatisfaction', 'EmpHourlyRate', 'EmpJobInvolvement', 'EmpJobSatisfaction', 'EmpRelationshipSatisfaction', 'EmpWorkLifeBalance', 'OverTime', 'EmpLastSalaryHikePercent', 'TrainingTimesLastYear', 'ExperienceYearsAtThisCompany', 'ExperienceYearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'Attrition']


**Two models will be trained in `train_model.ipynb`:**
1. **Hiring Model** — trained only on `hiring_safe_features` — this is the model that actually
   satisfies the brief's "used to hire employees" requirement.
2. **Diagnostic Model** — trained on all available features — used to answer "what drives
   performance across the current workforce" (the *Top 3 Factors* business question), which is
   a legitimate use of tenure/satisfaction data since it's about existing employees, not
   candidates.

## 5. Save Processed Dataset and Encoding Artifacts

In [8]:
os.makedirs('../../data/processed', exist_ok=True)
os.makedirs('../../models', exist_ok=True)

out_path = '../../data/processed/employee_performance_processed.csv'
df_processed.to_csv(out_path, index=False)
df.to_csv('../../data/processed/employee_performance_clean.csv', index=False)

# Persist the FITTED encoders (the actual objects, not just their printed mapping)
joblib.dump(label_encoders, '../../models/label_encoders.pkl')
joblib.dump(binary_maps, '../../models/binary_maps.pkl')
joblib.dump(hiring_safe_features, '../../models/hiring_safe_features.pkl')
joblib.dump(not_hiring_safe_features, '../../models/not_hiring_safe_features.pkl')

print('Saved processed data to:', out_path)
print('Saved encoders to: ../../models/label_encoders.pkl, binary_maps.pkl')
print('Saved feature-availability lists to: ../../models/hiring_safe_features.pkl, not_hiring_safe_features.pkl')
print('Final processed shape:', df_processed.shape)

Saved processed data to: ../../data/processed/employee_performance_processed.csv
Saved encoders to: ../../models/label_encoders.pkl, binary_maps.pkl
Saved feature-availability lists to: ../../models/hiring_safe_features.pkl, not_hiring_safe_features.pkl
Final processed shape: (1200, 27)


## Summary
- Loaded 1200 records × 28 columns, confirmed no missing values and no duplicates.
- Dropped the non-predictive identifier column (`EmpNumber`).
- Encoded 3 binary categoricals (Gender, OverTime, Attrition) and 5 nominal categoricals (EducationBackground, MaritalStatus, EmpDepartment, EmpJobRole, BusinessTravelFrequency) via label encoding — **and persisted the fitted encoders themselves** (`models/label_encoders.pkl`, `models/binary_maps.pkl`) so raw new records can be transformed consistently later, not just data that's already encoded.
- Classified every feature as hiring-safe or not, and saved both lists for reuse in training and prediction, directly addressing the model's intended hiring use case.
- Output: `data/processed/employee_performance_processed.csv` (modeling-ready) and `employee_performance_clean.csv` (human-readable, for EDA).